# Task 1: Data acquisition

How to download NHL play-by-play data (2016-17 to 2023-24), cache it locally, and load it back.
All the logic lives in `ift6758/data/nhl.py`; this notebook is the walkthrough you can adapt for the blog post.

**Before running:** copy `.env.example` to `.env` (or `export NHL_DATA_DIR=...`) to choose where raw JSON is cached.
The default is `data/raw/`, which is git-ignored. Never commit the data.

In [1]:
import json
from ift6758.data import (
    get_game, download_season, download_seasons, load_season,
    regular_season_game_id, playoff_game_id,
)

## 1. Game IDs

A game ID is `YYYY` + `TT` + `NNNN`: season start year, game type (`02` regular, `03` playoffs), then a game number.
For playoffs the last four digits are `0` + round + matchup + game number.

In [2]:
print(regular_season_game_id(2016, 1))   # first game of the 2016-17 regular season
print(playoff_game_id(2022, 4, 1, 1))    # 2022-23 Stanley Cup Final, game 1 -> 2022030411

2016020001
2022030411


## 2. Fetch one game

`get_game` checks the cache first. The first call downloads and saves the raw JSON; later calls read from disk without any request.

In [3]:
game = get_game("2022030411")
print(list(game.keys()))
print(len(game["plays"]), "events")
game["plays"][0]

['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock', 'displayPeriod', 'gameOutcome', 'plays', 'rosterSpots', 'regPeriods', 'summary']
408 events


{'eventId': 51,
 'periodDescriptor': {'number': 1,
  'periodType': 'REG',
  'maxRegulationPeriods': 3},
 'timeInPeriod': '00:00',
 'timeRemaining': '20:00',
 'situationCode': '1551',
 'homeTeamDefendingSide': 'right',
 'typeCode': 520,
 'typeDescKey': 'period-start',
 'sortOrder': 8}

## 3. Download a whole season

Regular season: counts up from game 1 and stops after a few consecutive 404s (season lengths differ).
Playoffs: walks round, matchup, game 1-7 and moves to the next series at the first missing game.
Re-running is free because everything is cached.

In [4]:
season = download_season(2016)
{k: len(v) for k, v in season.items()}

2016 regular: 1230game [15:49,  1.30game/s]
2016 playoffs: 87game [01:16,  1.14game/s]


{'regular': 1230, 'playoffs': 87}

## 4. Everything we need: 2016-17 to 2023-24

Slow the first time (be patient, someone else is paying for these requests), instant afterwards.

In [5]:
download_seasons(first=2016, last=2023)

2016 regular: 1230game [00:04, 265.76game/s] 
2016 playoffs: 87game [00:07, 11.45game/s]
2017 regular: 1271game [16:14,  1.30game/s]
2017 playoffs: 84game [01:13,  1.15game/s]
2018 regular: 1271game [16:14,  1.30game/s]
2018 playoffs: 87game [01:12,  1.20game/s]
2019 regular: 1082game [13:41,  1.32game/s]
2019 playoffs: 86game [01:12,  1.19game/s]
2020 regular: 868game [10:56,  1.32game/s]
2020 playoffs: 84game [01:11,  1.18game/s]
2021 regular: 1312game [16:38,  1.31game/s]
2021 playoffs: 89game [01:15,  1.18game/s]
2022 regular: 1312game [16:48,  1.30game/s]
2022 playoffs: 88game [01:14,  1.19game/s]
2023 regular: 1312game [16:38,  1.31game/s]
2023 playoffs: 88game [01:13,  1.19game/s]


## 5. Load from disk only

`load_season` never touches the network, which is what you want when building the tidy dataset.

In [6]:
regular = load_season(2016, "regular")
playoffs = load_season(2016, "playoffs")
len(regular), len(playoffs)

(1230, 87)

## 6. Sanity checks

Compare per-season counts against the NHL's published game counts before trusting the data.

In [7]:
for s in range(2016, 2024):
    print(s, len(load_season(s, "regular")), len(load_season(s, "playoffs")))

2016 1230 87
2017 1271 84
2018 1271 87
2019 1082 86
2020 868 84
2021 1312 89
2022 1312 88
2023 1312 88
